# 03 — Modeling: predicting stickiness

We train simple, interpretable models to predict the **sticky** label (top ~20% popularity) from audio features. This is **not** a claim that audio alone *causes* popularity — marketing, artist reach, and timing confound the outcome. The goal is **structure + interpretability**: which measured features align with the proxy label in this dataset?


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
import statsmodels.api as sm
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import features
from src import modeling
from src import visuals

FIG_DIR = PROJECT_ROOT / "outputs" / "figures"
TBL_DIR = PROJECT_ROOT / "outputs" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TBL_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "spotify_model_data.csv"
df = pd.read_csv(DATA_PATH)
print(df.shape)


## 1. Feature matrix & train/test split (80/20, stratified)


In [ ]:
INCLUDE_GENRE = False  # set True if you want one-hot genre features (sparse; needs enough rows per genre)

X, y = features.split_features_target(df, target_col="sticky", include_genre=INCLUDE_GENRE)
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_s, X_test_s, scaler = features.scale_train_test(X_train, X_test)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Features:", feature_names[:12], "...")


## 2. Logistic regression (standardized features)


In [ ]:
log_reg = modeling.train_logistic_regression(X_train_s, y_train, class_weight=None)
log_metrics = modeling.evaluate_classifier(log_reg, X_test_s, y_test)
print("Logistic regression metrics:", log_metrics)

fig = visuals.plot_logistic_coefficients(log_reg, feature_names, save_path=FIG_DIR / "09_logistic_coefficients.png")
plt.show()


## 3. Random forest classifier


In [ ]:
rf = modeling.train_random_forest(X_train, y_train, class_weight=None)
rf_metrics = modeling.evaluate_classifier(rf, X_test, y_test)
print("Random forest metrics:", rf_metrics)

importances = rf.feature_importances_
fig = visuals.plot_feature_importance(importances, feature_names, title="Random forest feature importance", save_path=FIG_DIR / "10_rf_feature_importance.png")
plt.show()


## 4. Model comparison


In [ ]:
comp = modeling.build_classification_report_df({
    "logistic_regression": log_metrics,
    "random_forest": rf_metrics,
})
display(comp.round(4))
comp.to_csv(TBL_DIR / "model_metrics.csv")


## 5. Confusion matrices


In [ ]:
fig = visuals.plot_confusion_matrix_from_model(log_reg, X_test_s, y_test, save_path=FIG_DIR / "11_confusion_matrix_logistic.png")
plt.show()
fig = visuals.plot_confusion_matrix_from_model(rf, X_test, y_test, save_path=FIG_DIR / "12_confusion_matrix_rf.png")
plt.show()


## 6. Optional: regression on standardized popularity (`popularity_z`)


In [ ]:
# Same train/test indices as classification; target is continuous standardized popularity.
if "popularity_z" not in df.columns:
    print("popularity_z missing — skip regression block.")
else:
    yz_train = df.loc[X_train.index, "popularity_z"]
    yz_test = df.loc[X_test.index, "popularity_z"]
    lin = modeling.train_linear_regression(X_train_s, yz_train)
    reg_metrics = modeling.evaluate_regression(lin, X_test_s, yz_test)
    print("Regression on popularity_z:", reg_metrics)


## 7. Optional: OLS on `popularity_z` (statsmodels, training split)


In [ ]:
# Coefficients with classic stats table; linear assumptions only approximate here.
if "popularity_z" not in df.columns:
    print("Skip OLS.")
else:
    y_pop = df.loc[X_train.index, "popularity_z"]
    X_ols = sm.add_constant(X_train_s, has_constant="add")
    ols = sm.OLS(y_pop, X_ols).fit()
    print(ols.summary().tables[1])


## 8. Error analysis: false positives & false negatives


In [ ]:
# Align test indices with metadata
test_idx = X_test.index
meta_cols = [c for c in ("track_name", "artist_name", "genre", "popularity") if c in df.columns]

y_pred_lr = log_reg.predict(X_test_s)
err = pd.DataFrame({
    "y_true": y_test,
    "y_pred": y_pred_lr,
}, index=test_idx)
for c in meta_cols:
    err[c] = df.loc[test_idx, c].values

fp = err[(err["y_true"] == 0) & (err["y_pred"] == 1)]
fn = err[(err["y_true"] == 1) & (err["y_pred"] == 0)]
print("False positives (sample up to 5):")
display(fp.head(5))
print("False negatives (sample up to 5):")
display(fn.head(5))


## 9. Conclusions & limitations

- **Interpretation:** Compare logistic coefficients (direction/magnitude on log-odds scale) with RF importance. Disagreement can happen — trees capture nonlinearity; linear models summarize average linear association on the standardized scale.
- **Limitations:** Popularity is an imperfect proxy for replay or stickiness; confounds include marketing, artist fanbase, and release effects. **Correlation is not causation.**
- **Product angle (suggestive):** Weak-but-interpretable signals can still inform **cold-start priors**, **playlist ordering heuristics**, or **risk scoring** when engagement logs are unavailable — always as a supplement to real user feedback.

Re-run after updating the raw CSV to refresh metrics and plots.
